# LeetCode 3283 — Maximum Number of Moves to Kill All Pawns bằng Python

- Link: https://leetcode.com/problems/maximum-number-of-moves-to-kill-all-pawns/
- Độ khó: **Hard**
- Chủ đề: **BFS mã** cho khoảng cách giữa các điểm quan trọng, **minimax trên bitmask** `(tập tốt còn lại, vị trí mã)`, lượt đi suy ra từ **số tốt đã bắt**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxMoves(self, kx: int, ky: int, positions: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Mỗi lượt, người chơi chọn một tốt còn lại; mã đi **đường ngắn nhất** tới đó (đi qua tốt khác không sao). Số nước của lượt = khoảng cách mã giữa vị trí hiện tại và tốt được chọn.
2. Chỉ `≤ 15` tốt + vị trí xuất phát ⇒ tính trước ma trận khoảng cách `16 × 16` bằng `16` lần **BFS mã** trên bàn `50 × 50`.
3. Trò chơi tổng hằng (Alice tối đa, Bob tối thiểu tổng nước) ⇒ **minimax**: `f(mask, i)` = tổng nước còn lại tối ưu khi tập tốt còn lại là `mask`, mã đang ở điểm `i`. Lượt của ai suy ra từ `n − popcount(mask)`.
4. Cài lặp theo `mask` tăng dần: `O(2^n · n²)` ≈ `7·10^6` phép.
5. Kiểm chứng bằng cây trò chơi vét cạn.

## 2. Đề bài & ví dụ

Bàn cờ `50 × 50` có một quân mã tại `(kx, ky)` và các tốt `positions`. Alice đi trước; mỗi lượt người chơi chọn **một** tốt còn trên bàn và bắt nó bằng mã với **số nước ít nhất** (có thể chọn tốt không phải gần nhất). Alice muốn **tối đa** tổng số nước của cả hai đến khi hết tốt, Bob muốn **tối thiểu**. Trả về tổng số nước khi cả hai chơi tối ưu.

**Ví dụ 1:** `kx = 1, ky = 1`, `[[0,0]]` → `4`.
**Ví dụ 2:** `kx = 0, ky = 2`, `[[1,1],[2,2],[3,3]]` → `8`.
**Ví dụ 3:** `kx = 0, ky = 0`, `[[1,2],[2,4]]` → `3`.

**Ràng buộc:** `0 ≤ kx, ky ≤ 49`; `1 ≤ len(positions) ≤ 15`; vị trí phân biệt, không trùng mã.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache

examples = [(1, 1, [[0, 0]], 4), (0, 2, [[1, 1], [2, 2], [3, 3]], 8), (0, 0, [[1, 2], [2, 4]], 3)]
for kx, ky, pos, exp in examples:
    print((kx, ky), pos, '-> kỳ vọng', exp)

(1, 1) [[0, 0]] -> kỳ vọng 4
(0, 2) [[1, 1], [2, 2], [3, 3]] -> kỳ vọng 8
(0, 0) [[1, 2], [2, 4]] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. BFS của quân mã

In [2]:
MOVES = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]

def knight_bfs(sx, sy, N=50):
    d = [[-1] * N for _ in range(N)]
    d[sx][sy] = 0
    q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        for dx, dy in MOVES:
            a, b = x + dx, y + dy
            if 0 <= a < N and 0 <= b < N and d[a][b] < 0:
                d[a][b] = d[x][y] + 1
                q.append((a, b))
    return d

d = knight_bfs(1, 1)
print('khoảng cách mã từ (1,1) tới (0,0):', d[0][0], '| tới (2,2):', d[2][2], '| tới (49,49):', d[49][49])

khoảng cách mã từ (1,1) tới (0,0): 4 | tới (2,2): 2 | tới (49,49): 32


Lưu ý `(1, 1) → (0, 0)` cần **4** nước vì bị chặn bởi mép bàn — không thể dùng công thức khoảng cách mã trên bàn vô hạn một cách ngây thơ.

### 3.2. Ma trận khoảng cách giữa các điểm quan trọng

Điểm `0..n−1` là các tốt, điểm `n` là vị trí xuất phát. `D[i][j]` = khoảng cách mã — `n + 1` lần BFS, mỗi lần `O(2500 · 8)`.

In [3]:
def distance_matrix(kx, ky, pos):
    pts = [tuple(p) for p in pos] + [(kx, ky)]
    D = []
    for x, y in pts:
        d = knight_bfs(x, y)
        D.append([d[a][b] for a, b in pts])
    return D

for row in distance_matrix(0, 2, [[1, 1], [2, 2], [3, 3]]):
    print(row)

[0, 2, 4, 2]
[2, 0, 2, 2]
[4, 2, 0, 2]
[2, 2, 2, 0]


### 3.3. Lượt đi suy ra từ trạng thái

Sau khi đã bắt `c = n − popcount(mask)` tốt: `c` chẵn ⇒ lượt Alice, `c` lẻ ⇒ lượt Bob. Không cần lưu thêm biến lượt.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 (ma trận ở trên, điểm 3 = xuất phát `(0, 2)`): Alice cân nhắc từng tốt đầu tiên và giả định Bob đáp trả tối ưu:

- Bắt `(2,2)` (2 nước): Bob đứng ở `(2,2)` chọn tốt tiếp theo sao cho **tổng còn lại** nhỏ nhất (không nhất thiết là tốt gần nhất), rồi Alice bắt nốt. Như lời giải của đề: Bob bắt `(3,3)` (2 nước), Alice bắt `(1,1)` (4 nước) ⇒ `2 + 2 + 4 = 8`.
- Các lựa chọn khác cho tổng nhỏ hơn ⇒ đáp án `8`.

Tham lam "Alice chọn xa nhất, Bob chọn gần nhất" **không** đủ — cần xét toàn bộ tương lai (minimax).

## 5. Cách 1 — Brute Force: cây trò chơi đầy đủ (không nhớ)

In [4]:
def brute(kx, ky, pos):
    n = len(pos)
    D = distance_matrix(kx, ky, pos)
    def play(remaining, cur, alice):
        if not remaining:
            return 0
        vals = [D[cur][j] + play(remaining - {j}, j, not alice) for j in remaining]
        return max(vals) if alice else min(vals)
    return play(frozenset(range(n)), n, True)

for kx, ky, pos, exp in examples:
    print(brute(kx, ky, pos), '| kỳ vọng', exp)

4 | kỳ vọng 4
8 | kỳ vọng 8
3 | kỳ vọng 3


**Độ phức tạp:** `O(n!)` — chỉ cho `n ≤ 7`.

## 6. Ý tưởng thuật toán / bước đột phá

```
f(0, i) = 0
f(mask, i) = max_{j ∈ mask} D[i][j] + f(mask \ {j}, j)   nếu lượt Alice
           = min_{j ∈ mask} D[i][j] + f(mask \ {j}, j)   nếu lượt Bob
đáp án = f(toàn bộ, n)
```

`mask \ {j}` nhỏ hơn `mask` ⇒ duyệt `mask` tăng dần. Chỉ cần các `i ∉ mask` (mã đang đứng ở tốt đã bắt hoặc ở điểm xuất phát).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ

In [5]:
def memo_solution(kx, ky, pos):
    n = len(pos)
    D = distance_matrix(kx, ky, pos)

    @lru_cache(maxsize=None)
    def f(mask, i):
        if mask == 0:
            return 0
        alice = (n - bin(mask).count('1')) % 2 == 0
        best = None
        m = mask
        while m:
            j = (m & -m).bit_length() - 1
            m &= m - 1
            v = D[i][j] + f(mask ^ (1 << j), j)
            if best is None or (v > best if alice else v < best):
                best = v
        return best

    ans = f((1 << n) - 1, n)
    f.cache_clear()
    return ans

for kx, ky, pos, exp in examples:
    print(memo_solution(kx, ky, pos), '| kỳ vọng', exp)

4 | kỳ vọng 4
8 | kỳ vọng 8
3 | kỳ vọng 3


### 7.2. Cách 3 — DP lặp theo bitmask (lời giải chính)

In [6]:
def iterative_solution(kx, ky, pos):
    n = len(pos)
    D = distance_matrix(kx, ky, pos)
    f = [[0] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        alice = (n - bin(mask).count('1')) % 2 == 0
        bits = [j for j in range(n) if mask >> j & 1]
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue                    # mã không thể đứng ở tốt chưa bắt
            Di = D[i]
            vals = [Di[j] + f[mask ^ (1 << j)][j] for j in bits]
            f[mask][i] = max(vals) if alice else min(vals)
    return f[(1 << n) - 1][n]

for kx, ky, pos, exp in examples:
    print(iterative_solution(kx, ky, pos), '| kỳ vọng', exp)

4 | kỳ vọng 4
8 | kỳ vọng 8
3 | kỳ vọng 3


### 7.3. Cách 4 — Alpha–beta trên cây trò chơi (minh họa cắt tỉa)

Không nhớ trạng thái, nhưng cắt các nhánh không thể ảnh hưởng kết quả. Vẫn mũ trong trường hợp xấu — dùng để thấy vì sao **nhớ trạng thái** (cách 2, 3) mới là chìa khóa.

In [7]:
def alphabeta(kx, ky, pos):
    n = len(pos)
    D = distance_matrix(kx, ky, pos)
    nodes = [0]
    def ab(remaining, cur, alice, alpha, beta):
        nodes[0] += 1
        if not remaining:
            return 0
        order = sorted(remaining, key=lambda j: -D[cur][j] if alice else D[cur][j])
        if alice:
            v = -1
            for j in order:
                v = max(v, D[cur][j] + ab(remaining - {j}, j, False, alpha - D[cur][j], beta - D[cur][j]))
                alpha = max(alpha, v)
                if alpha >= beta:
                    break
            return v
        v = 10 ** 9
        for j in order:
            v = min(v, D[cur][j] + ab(remaining - {j}, j, True, alpha - D[cur][j], beta - D[cur][j]))
            beta = min(beta, v)
            if alpha >= beta:
                break
        return v
    return ab(frozenset(range(n)), n, True, -10 ** 9, 10 ** 9), nodes[0]

for kx, ky, pos, exp in examples:
    print(alphabeta(kx, ky, pos), '| kỳ vọng', exp)

(4, 2) | kỳ vọng 4
(8, 14) | kỳ vọng 8
(3, 5) | kỳ vọng 3


Khi gọi đệ quy, cửa sổ `(α, β)` được **dịch** đi `D[cur][j]` vì giá trị con được cộng thêm khoảng cách của nước hiện tại.

## 8. TRACE từng bước

Trace ví dụ 2: lấy chuỗi lựa chọn tối ưu từ bảng DP.

In [8]:
def trace(kx, ky, pos):
    n = len(pos)
    D = distance_matrix(kx, ky, pos)
    f = [[0] * (n + 1) for _ in range(1 << n)]
    choice = [[-1] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        alice = (n - bin(mask).count('1')) % 2 == 0
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue
            best = None
            for j in range(n):
                if mask >> j & 1:
                    v = D[i][j] + f[mask ^ (1 << j)][j]
                    if best is None or (v > best if alice else v < best):
                        best, choice[mask][i] = v, j
            f[mask][i] = best
    mask, cur, turn = (1 << n) - 1, n, 0
    pts = [tuple(p) for p in pos] + [(kx, ky)]
    while mask:
        j = choice[mask][cur]
        print(f"{'Alice' if turn % 2 == 0 else 'Bob  '} bắt {pts[j]} từ {pts[cur]}: {D[cur][j]} nước (còn lại tối ưu {f[mask][cur]})")
        mask ^= 1 << j; cur = j; turn += 1
    print('tổng:', f[(1 << n) - 1][n], '| vét cạn:', brute(kx, ky, pos))

trace(0, 2, [[1, 1], [2, 2], [3, 3]])

Alice bắt (2, 2) từ (0, 2): 2 nước (còn lại tối ưu 8)
Bob   bắt (1, 1) từ (2, 2): 2 nước (còn lại tối ưu 6)
Alice bắt (3, 3) từ (1, 1): 4 nước (còn lại tối ưu 4)
tổng: 8 | vét cạn: 8


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxMoves(self, kx: int, ky: int, positions: List[List[int]]) -> int:
        n = len(positions)
        pts = [tuple(p) for p in positions] + [(kx, ky)]
        moves = ((1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1))
        D = []
        for sx, sy in pts:
            d = [[-1] * 50 for _ in range(50)]
            d[sx][sy] = 0
            q = [(sx, sy)]
            for x, y in q:
                nd = d[x][y] + 1
                for dx, dy in moves:
                    a, b = x + dx, y + dy
                    if 0 <= a < 50 and 0 <= b < 50 and d[a][b] < 0:
                        d[a][b] = nd
                        q.append((a, b))
            D.append([d[a][b] for a, b in pts])
        full = (1 << n) - 1
        f = [[0] * (n + 1) for _ in range(full + 1)]
        for mask in range(1, full + 1):
            alice = (n - bin(mask).count('1')) % 2 == 0
            bits = [j for j in range(n) if mask >> j & 1]
            subs = [f[mask ^ (1 << j)] for j in bits]
            row = f[mask]
            for i in range(n + 1):
                if i < n and mask >> i & 1:
                    continue
                Di = D[i]
                if alice:
                    best = -1
                    for j, sub in zip(bits, subs):
                        v = Di[j] + sub[j]
                        if v > best:
                            best = v
                else:
                    best = 1 << 30
                    for j, sub in zip(bits, subs):
                        v = Di[j] + sub[j]
                        if v < best:
                            best = v
                row[i] = best
        return f[full][n]

for kx, ky, pos, exp in examples:
    got = Solution().maxMoves(kx, ky, pos)
    print(got, 'OK' if got == exp else 'SAI')

4 OK


8 OK
3 OK


## 10. Vì sao thuật toán đúng

1. **Số nước một lượt** = khoảng cách mã ngắn nhất (đề bắt buộc đi đường ngắn nhất; tốt khác không cản đường) = `D[cur][j]` tính bằng BFS (đồ thị không trọng số).
2. **Trạng thái đủ:** tương lai chỉ phụ thuộc tập tốt còn lại và vị trí mã (vị trí = tốt vừa bắt, hoặc điểm xuất phát). Lượt suy ra từ số tốt đã bắt.
3. **Minimax:** trò chơi hữu hạn, thông tin đầy đủ, tổng hằng ⇒ giá trị tối ưu thỏa phương trình Bellman ở mục 6 (quy nạp theo `popcount(mask)`).
4. **Thứ tự tính:** `mask ^ (1 << j) < mask` ⇒ đã tính khi duyệt `mask` tăng dần.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Cây trò chơi | `O(n!)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(2^n · n²)` | `O(2^n · n)` |
| 3. DP lặp | `O(2^n · n² + n · 50² · 8)` | `O(2^n · n)` |
| 4. Alpha–beta | mũ (ít nút hơn cách 1) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'memo': memo_solution, 'iterative': iterative_solution, 'alphabeta': lambda a, b, c: alphabeta(a, b, c)[0],
         'submit': lambda a, b, c: Solution().maxMoves(a, b, c)}
edge = [(0, 0, [[1, 2]], 1), (0, 0, [[49, 49]], knight_bfs(0, 0)[49][49]), (25, 25, [[25, 26]], 3), (0, 0, [[0, 1], [1, 0]], None)]
for kx, ky, pos, exp in examples + edge:
    e = brute(kx, ky, pos)
    if exp is not None:
        assert e == exp, (kx, ky, pos, e)
    for name, f in impls.items():
        assert f(kx, ky, pos) == e, (name, kx, ky, pos)
print('Ví dụ đề và ca biên: tất cả đúng. Ca (0,0) với hai tốt kề góc:', brute(0, 0, [[0, 1], [1, 0]]))

Ví dụ đề và ca biên: tất cả đúng. Ca (0,0) với hai tốt kề góc: 5


### 12.2. Đối chiếu ngẫu nhiên với cây trò chơi

In [11]:
import random
rng = random.Random(3283)
cells = [(x, y) for x in range(50) for y in range(50)]
cnt = 0
for _ in range(60):
    n = rng.randint(1, 6)
    pick = rng.sample(cells, n + 1)
    kx, ky = pick[-1]
    pos = [list(p) for p in pick[:-1]]
    exp = brute(kx, ky, pos)
    for name, f in impls.items():
        assert f(kx, ky, pos) == exp, (name, kx, ky, pos)
    cnt += 1
print('Khớp cây trò chơi trên', cnt, 'bàn ngẫu nhiên.')

Khớp cây trò chơi trên 60 bàn ngẫu nhiên.


### 12.3. Ca lớn: 15 tốt

In [12]:
import time
for trial in range(3):
    pick = rng.sample(cells, 16)
    kx, ky = pick[-1]; pos = [list(p) for p in pick[:-1]]
    t = time.time(); a = Solution().maxMoves(kx, ky, pos); t1 = time.time() - t
    t = time.time(); b = memo_solution(kx, ky, pos); t2 = time.time() - t
    assert a == b
    print(f"bàn {trial}: {a:4d} nước | nộp {t1:.2f}s | đệ quy có nhớ {t2:.2f}s")

bàn 0:  184 nước | nộp 0.35s | đệ quy có nhớ 0.93s


bàn 1:  175 nước | nộp 0.44s | đệ quy có nhớ 1.36s


bàn 2:  165 nước | nộp 0.49s | đệ quy có nhớ 1.06s


Kiểm tra độc lập (tính chất): đáp án nằm giữa "Bob chọn hết" (min mọi thứ tự) và "Alice chọn hết" (max mọi thứ tự) — tính được bằng DP một người chơi; với 1 tốt thì đúng bằng khoảng cách.

In [13]:
def single_player(kx, ky, pos, better):
    n = len(pos); D = distance_matrix(kx, ky, pos)
    f = [[0] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue
            f[mask][i] = better(D[i][j] + f[mask ^ (1 << j)][j] for j in range(n) if mask >> j & 1)
    return f[(1 << n) - 1][n]

for _ in range(5):
    pick = rng.sample(cells, 9); kx, ky = pick[-1]; pos = [list(p) for p in pick[:-1]]
    lo, hi, v = single_player(kx, ky, pos, min), single_player(kx, ky, pos, max), Solution().maxMoves(kx, ky, pos)
    assert lo <= v <= hi
print('min-thứ-tự <= đáp án <= max-thứ-tự: đúng trên 5 bàn 8 tốt.')

min-thứ-tự <= đáp án <= max-thứ-tự: đúng trên 5 bàn 8 tốt.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: Alice chọn tốt xa nhất, Bob chọn tốt gần nhất

In [14]:
def wrong_greedy(kx, ky, pos):
    n = len(pos); D = distance_matrix(kx, ky, pos)
    rem, cur, total, alice = set(range(n)), n, 0, True
    while rem:
        j = (max if alice else min)(rem, key=lambda t: D[cur][t])      # SAI: chỉ nhìn một bước
        total += D[cur][j]; rem.remove(j); cur = j; alice = not alice
    return total

found = None
r_ = random.Random(1)
for _ in range(3000):
    pick = r_.sample([(x, y) for x in range(6) for y in range(6)], r_.randint(3, 5))
    kx, ky = pick[-1]; pos = [list(p) for p in pick[:-1]]
    if wrong_greedy(kx, ky, pos) != brute(kx, ky, pos):
        found = (kx, ky, pos); break
print('phản ví dụ:', found, '-> sai:', wrong_greedy(*found), '| đúng:', Solution().maxMoves(*found))

phản ví dụ: (5, 3, [[2, 1], [4, 3], [0, 1]]) -> sai: 7 | đúng: 9


**Vì sao sai:** chọn xa nhất ngay có thể đặt mã vào chỗ để Bob bắt tiếp rất rẻ; minimax phải nhìn toàn bộ tương lai.

### Lỗi 2 — Dùng khoảng cách Chebyshev/Manhattan "đổi ra" nước mã

In [15]:
def wrong_distance(kx, ky, pos):
    n = len(pos); pts = [tuple(p) for p in pos] + [(kx, ky)]
    D = [[(abs(a[0] - b[0]) + abs(a[1] - b[1]) + 2) // 3 for b in pts] for a in pts]   # SAI: xấp xỉ
    f = [[0] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        alice = (n - bin(mask).count('1')) % 2 == 0
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue
            vals = [D[i][j] + f[mask ^ (1 << j)][j] for j in range(n) if mask >> j & 1]
            f[mask][i] = max(vals) if alice else min(vals)
    return f[(1 << n) - 1][n]

print('sai :', wrong_distance(1, 1, [[0, 0]]), '| đúng:', Solution().maxMoves(1, 1, [[0, 0]]))

sai : 1 | đúng: 4


**Vì sao sai:** khoảng cách mã không có công thức đơn giản (đặc biệt gần mép/góc: `(1,1) → (0,0)` mất 4 nước). Bàn chỉ 2500 ô — BFS là cách chắc chắn.

### Lỗi 3 — Đảo lượt (cho Bob đi trước)

In [16]:
def wrong_turn(kx, ky, pos):
    n = len(pos); D = distance_matrix(kx, ky, pos)
    f = [[0] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        alice = (n - bin(mask).count('1')) % 2 == 1               # SAI: lệch lượt
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue
            vals = [D[i][j] + f[mask ^ (1 << j)][j] for j in range(n) if mask >> j & 1]
            f[mask][i] = max(vals) if alice else min(vals)
    return f[(1 << n) - 1][n]

print('sai :', wrong_turn(0, 0, [[1, 2], [2, 4]]), '| đúng:', Solution().maxMoves(0, 0, [[1, 2], [2, 4]]))

sai : 2 | đúng: 3


**Vì sao sai:** Alice đi trước (`0` tốt đã bắt ⇒ lượt Alice). Đảo lượt cho Bob quyền chọn trước.

### Lỗi 4 — Luôn tính khoảng cách từ vị trí **xuất phát** (quên mã đã di chuyển)

In [17]:
def wrong_static_knight(kx, ky, pos):
    n = len(pos); D = distance_matrix(kx, ky, pos)
    @lru_cache(maxsize=None)
    def f(mask):
        if mask == 0:
            return 0
        alice = (n - bin(mask).count('1')) % 2 == 0
        vals = [D[n][j] + f(mask ^ (1 << j)) for j in range(n) if mask >> j & 1]   # SAI: luôn từ điểm n
        return max(vals) if alice else min(vals)
    return f((1 << n) - 1)

print('sai :', wrong_static_knight(0, 2, [[1, 1], [2, 2], [3, 3]]), '| đúng:', Solution().maxMoves(0, 2, [[1, 1], [2, 2], [3, 3]]))

sai : 6 | đúng: 8


**Vì sao sai:** sau mỗi lượt, mã đứng tại ô của tốt vừa bắt; lượt sau đi từ đó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "≤ 15–20 mục tiêu, thứ tự thăm quan trọng" ⇒ bitmask DP (kiểu TSP).
- "Hai người chơi tối ưu, một người max, một người min tổng" ⇒ minimax; lượt suy ra từ số bước đã đi.
- Khoảng cách trên lưới với quân cờ ⇒ BFS từ mỗi điểm quan trọng (đừng đoán công thức).

## 15. Biến thể của bài

1. Chỉ một người chơi (Alice chọn hết) ⇒ TSP đường đi "dài nhất" — `single_player(…, max)`.
2. Có **chướng ngại** trên bàn ⇒ BFS tránh ô cấm, có thể không tới được (khoảng cách ∞).
3. Quân khác (vua, tượng) ⇒ đổi tập nước đi trong BFS.
4. Mỗi người tối đa **điểm riêng** của mình (không tổng hằng) ⇒ cần giá trị vector, không còn minimax đơn.

In [18]:
# Biến thể 3: quân vua (khoảng cách Chebyshev) — kiểm tra BFS tổng quát
def piece_bfs(sx, sy, moves, N=50):
    d = [[-1] * N for _ in range(N)]; d[sx][sy] = 0; q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        for dx, dy in moves:
            a, b = x + dx, y + dy
            if 0 <= a < N and 0 <= b < N and d[a][b] < 0:
                d[a][b] = d[x][y] + 1; q.append((a, b))
    return d

KING = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if dx or dy]
dk = piece_bfs(3, 7, KING)
assert all(dk[x][y] == max(abs(x - 3), abs(y - 7)) for x in range(50) for y in range(50))
# Biến thể 2: chướng ngại
def knight_bfs_blocked(sx, sy, blocked, N=50):
    d = [[-1] * N for _ in range(N)]; d[sx][sy] = 0; q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        for dx, dy in MOVES:
            a, b = x + dx, y + dy
            if 0 <= a < N and 0 <= b < N and d[a][b] < 0 and (a, b) not in blocked:
                d[a][b] = d[x][y] + 1; q.append((a, b))
    return d
db = knight_bfs_blocked(0, 0, {(1, 2), (2, 1)})
print('vua: BFS = Chebyshev ✓ | mã ở góc bị chặn cả hai nước -> (5,5) không tới được:', db[5][5] == -1)

vua: BFS = Chebyshev ✓ | mã ở góc bị chặn cả hai nước -> (5,5) không tới được: True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Cây trò chơi | `O(n!)` | Oracle |
| 2 | Đệ quy có nhớ | `O(2^n n²)` | Dễ viết |
| 3 | DP lặp bitmask | `O(2^n n²)` | **Nộp bài** |
| 4 | Alpha–beta | mũ | Minh họa cắt tỉa |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `kx = 0, ky = 0`, `positions = [[2, 1]]`.

**Bài 2 (tính tay):** `kx = 0, ky = 0`, `positions = [[1, 2], [3, 3]]`.

**Bài 3 (điền TODO):** trả về **chuỗi lựa chọn** tối ưu của hai người.

**Bài 4 (tự viết lại):** biến thể 1 — một người chơi muốn **tối thiểu** tổng (TSP đường đi).

**Bài 5 (mini challenge):** đáp án khi **Bob đi trước** — so sánh với Alice đi trước trên nhiều bàn.

In [19]:
def optimal_line_todo(kx, ky, pos):
    # TODO
    raise NotImplementedError

def min_total_todo(kx, ky, pos):
    # TODO
    raise NotImplementedError

def bob_first_todo(kx, ky, pos):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một nước mã. (2) Không còn tốt khác. (3) `1`.

**Bài 2.** (1) `D(start, (1,2)) = 1`, `D(start, (3,3)) = 2`, `D((1,2),(3,3)) = ?` (BFS). (2) Alice thử cả hai thứ tự, Bob không còn lựa chọn. (3) Lấy max.

**Bài 3.** (1) Lưu `choice[mask][i]`. (2) Đi theo `choice` từ trạng thái đầu. (3) Xem hàm `trace`.

**Bài 4.** (1) `single_player(…, min)`. (2) Kiểm bằng hoán vị. (3) `n ≤ 7`.

**Bài 5.** (1) Đổi điều kiện lượt. (2) Đó chính là "lỗi 3" — nhưng là câu trả lời đúng cho đề khác. (3) So sánh hai giá trị bằng thực nghiệm trên nhiều bàn ngẫu nhiên (không có chứng minh tổng quát trong notebook này).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 4
from itertools import permutations
assert Solution().maxMoves(0, 0, [[2, 1]]) == 1
assert Solution().maxMoves(0, 0, [[1, 2], [3, 3]]) == brute(0, 0, [[1, 2], [3, 3]])

def min_total(kx, ky, pos):
    return single_player(kx, ky, pos, min)

rng = random.Random(3)
for _ in range(20):
    pick = rng.sample(cells, rng.randint(2, 6)); kx, ky = pick[-1]; pos = [list(p) for p in pick[:-1]]
    D = distance_matrix(kx, ky, pos); n = len(pos)
    exp = min(sum(D[a][b] for a, b in zip((n,) + perm, perm)) for perm in permutations(range(n)))
    assert min_total(kx, ky, pos) == exp
print('Bài 1: 1 | Bài 2:', Solution().maxMoves(0, 0, [[1, 2], [3, 3]]), '| Bài 4 khớp duyệt hoán vị.')

Bài 1: 1 | Bài 2: 3 | Bài 4 khớp duyệt hoán vị.


In [21]:
# Bài 3, 5
def optimal_line(kx, ky, pos):
    n = len(pos); D = distance_matrix(kx, ky, pos)
    f = [[0] * (n + 1) for _ in range(1 << n)]; ch = [[-1] * (n + 1) for _ in range(1 << n)]
    for mask in range(1, 1 << n):
        alice = (n - bin(mask).count('1')) % 2 == 0
        for i in range(n + 1):
            if i < n and mask >> i & 1:
                continue
            best = None
            for j in range(n):
                if mask >> j & 1:
                    v = D[i][j] + f[mask ^ (1 << j)][j]
                    if best is None or (v > best if alice else v < best):
                        best, ch[mask][i] = v, j
            f[mask][i] = best
    line, mask, cur = [], (1 << n) - 1, n
    while mask:
        j = ch[mask][cur]; line.append(tuple(pos[j])); mask ^= 1 << j; cur = j
    return line, f[(1 << n) - 1][n]

def bob_first(kx, ky, pos):
    return wrong_turn(kx, ky, pos)

line, total = optimal_line(0, 2, [[1, 1], [2, 2], [3, 3]])
pts_line = [(0, 2)] + line
assert sum(knight_bfs(*a)[b[0]][b[1]] for a, b in zip(pts_line, pts_line[1:])) == total == 8
diffs = set()
for _ in range(10):
    pick = rng.sample(cells, 6); kx, ky = pick[-1]; pos = [list(p) for p in pick[:-1]]
    diffs.add((Solution().maxMoves(kx, ky, pos) > bob_first(kx, ky, pos)) - (Solution().maxMoves(kx, ky, pos) < bob_first(kx, ky, pos)))
print('Bài 3: chuỗi tối ưu', line, '| Bài 5: dấu của (Alice trước − Bob trước) gặp được:', sorted(diffs))

Bài 3: chuỗi tối ưu [(2, 2), (1, 1), (3, 3)] | Bài 5: dấu của (Alice trước − Bob trước) gặp được: [1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
D = BFS mã từ mỗi tốt và từ vị trí xuất phát (n + 1 lần)
f[0][i] = 0
for mask tăng dần: alice = (n - popcount(mask)) % 2 == 0
    for i ∉ mask (hoặc i = n): f[mask][i] = max/min_{j ∈ mask} D[i][j] + f[mask ^ 1<<j][j]
return f[full][n]
```

**Checklist:**

- [ ] Khoảng cách bằng BFS mã (có mép bàn)?
- [ ] Mã di chuyển tới ô tốt vừa bắt?
- [ ] Alice đi trước (0 tốt đã bắt)?
- [ ] Minimax trên toàn bộ tương lai (không tham lam)?
- [ ] Duyệt `mask` tăng dần?